# RSNA Knee: упаковка кэша в один файл

## Зачем

Билдер `03_build_cache_kaggle.ipynb` сохраняет часть кэша как ~2200 мелких файлов `.npz`. Кнопкой Download
такой датасет Kaggle часто не отдаёт («Data not available. This dataset may need its archive recreated»).
Этот ноутбук складывает одну часть в **один файл** `rsna_raw_partK.pack`, а один файл в Data Explorer
скачивается по прямой ссылке.

| Вопрос | Ответ |
|---|---|
| Меняются ли данные? | Нет. Файлы кладутся в архив как есть |
| Почему без сжатия? | `.npz` уже сжаты; второй раз сжимать - только время терять |
| Что такое `.pack`? | Обычный архив tar, но с другим расширением. Файлы `.tar` и `.zip` Kaggle при создании датасета распаковывает сам, и архив снова превратился бы в тысячи файлов |
| Почему отдельный ноутбук на каждую часть? | Из одного ноутбука Kaggle разрешает сделать только один датасет |

## Как запустить

1. Kaggle -> New Notebook -> File -> Import Notebook -> этот файл.
2. Input: датасет с частью кэша (тот, где лежит `index_part0.csv` или `index_part1.csv`). Можно подключить оба.
3. В первой ячейке `PART` = номер части, которую упаковываем (как в `index_partK.csv`).
4. Accelerator None -> **Save Version -> Save & Run All** (около 10-15 минут).
5. Output -> **New Dataset**. Для второй части: **Copy & Edit** этого ноутбука, другой `PART`, то же самое.

## Как скачать и распаковать на Mac

Data Explorer -> `rsna_raw_partK.pack` -> скачать. Потом распаковать в `data/raw/` (флаг `-x` - распаковать,
`-f` - из этого файла, `-C` - в эту папку):

```bash
tar -xf rsna_raw_part0.pack -C rsna-knee-abnormality-detection/data/raw
```

Если браузер открывает прямую ссылку, её можно отдать в `curl` и распаковывать прямо по ходу загрузки,
не храня архив на диске:

```bash
curl -L "<ссылка>" | tar -x -C rsna-knee-abnormality-detection/data/raw
```

## Настройки

| Параметр | Что значит |
|---|---|
| `PART` | какую часть упаковываем: 0 или 1 (номер из имени `index_partK.csv`) |
| `INPUT` | куда Kaggle подключает датасеты из Input |
| `OUT_DIR` | всё, что ноутбук пишет сюда, становится Output (и потом датасетом) |

In [ ]:
import json
import tarfile          # работа с архивами tar (стандартная библиотека Python)
import time
from pathlib import Path

PART = 0                                   # какую часть упаковываем: 0 или 1 (как в index_partK.csv)
INPUT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working")
T0 = time.time()


def log(msg):
    # печать с отметкой времени от старта: [  12.3s] сообщение
    print(f"[{time.time() - T0:6.1f}s] {msg}", flush=True)

## Находим часть кэша и проверяем, всё ли на месте

1. Обходим все подключённые датасеты (`os.walk`) и ищем файлы `index_part*.csv`. Печатаем, что нашли:
   если `PART` не тот, сразу видно, какие части вообще подключены.
2. Папка, где лежит `index_part{PART}.csv`, и есть корень части: рядом `studies/`, `rsna_raw_meta.json`, `train.csv`.
3. Сверяем: каждое колено из индекса должно иметь свой файл в `studies/`. Если чего-то нет, ноутбук
   остановится **до** упаковки, чтобы не скачивать неполный архив.

`followlinks=True` нужен, потому что Kaggle подключает датасеты через символические ссылки, а без этого
флага `os.walk` в них не заходит.

In [ ]:
import os

# ищем во всех подключённых датасетах (os.walk заходит и в смонтированные, и в связанные папки)
# os.walk отдаёт тройки (папка, подпапки, файлы); берём все файлы вида index_part*.csv
found = sorted(Path(root) / f for root, _, fs in os.walk(INPUT, followlinks=True)
               for f in fs if f.startswith("index_part") and f.endswith(".csv"))
print("подключено в Input:", sorted(p.name for p in INPUT.iterdir()) if INPUT.is_dir() else "ничего")
print("найдены части кэша:", [f"{p.name} в {p.parent}" for p in found] or "ни одной")
hits = [p for p in found if p.name == f"index_part{PART}.csv"]
assert hits, (f"не найден index_part{PART}.csv. Поставьте PART равным номеру из найденных выше "
              f"index_partK.csv или подключите датасет с нужной частью")
ROOT = hits[0].parent                      # корень части: тут studies/, index, meta, train.csv
log(f"часть {PART} найдена в {ROOT}")

import pandas as pd
index = pd.read_csv(ROOT / f"index_part{PART}.csv")              # одна строка = одна серия
expected = sorted(index["StudyInstanceUID"].astype(str).unique())  # какие колена должны быть
files = sorted((ROOT / "studies").glob("*.npz"))                  # какие файлы реально есть
have = {f.stem for f in files}                                     # f.stem = имя без .npz = ID колена
missing = [s for s in expected if s not in have]
size_gb = sum(f.stat().st_size for f in files) / 1024 ** 3
log(f"в индексе {len(expected)} исследований, в studies/ {len(files)} файлов, {size_gb:.2f} ГБ, не хватает {len(missing)}")
assert not missing, f"в датасете нет {len(missing)} файлов (например {missing[:3]}): часть собрана не полностью"

## Упаковка

`tarfile.open(out, "w")` создаёт архив без сжатия (`"w:gz"` было бы со сжатием). `tar.add(файл, arcname=...)`
кладёт файл внутрь под именем `arcname`. Имена внутри архива такие же, как должны лежать локально:
`studies/<ID>.npz`, `index_partK.csv` и т.д. Поэтому распаковка в `data/raw/` сразу даёт нужную структуру.

В конце архив открывается заново и пересчитываются файлы `studies/` внутри: их должно быть столько же, сколько упаковали.

In [ ]:
# Расширение .pack, а не .tar: при создании датасета Kaggle сам распаковывает .tar и .zip,
# и архив снова превратился бы в тысячи файлов. Внутри это обычный tar.
out = OUT_DIR / f"rsna_raw_part{PART}.pack"
with tarfile.open(out, "w") as tar:                     # без сжатия: npz уже сжаты
    for name in (f"index_part{PART}.csv", "rsna_raw_meta.json", "train.csv"):   # файлы-справки
        if (ROOT / name).is_file():
            tar.add(ROOT / name, arcname=name)
    for i, f in enumerate(files, 1):                    # enumerate(..., 1): считаем с 1, а не с 0
        tar.add(f, arcname=f"studies/{f.name}")
        if i % 500 == 0 or i == len(files):             # прогресс каждые 500 файлов
            log(f"  упаковано {i}/{len(files)}")
log(f"готово: {out.name}, {out.stat().st_size / 1024 ** 3:.2f} ГБ")

with tarfile.open(out) as tar:                          # проверка: в архиве все файлы
    n = sum(1 for m in tar.getmembers() if m.name.startswith("studies/"))
assert n == len(files), (n, len(files))
log(f"проверка архива: {n} исследований внутри")